<a href="https://colab.research.google.com/github/parthag1201/RAG-ify/blob/main/rag_from_scratch_P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ** Problem statement **
### It is hard to properly get semantic searches on embeddeding.
### User queries are a challenge. If user gives ambiguous queries, they'll get ambiguous matches

# ** Solutions **

### 1. Sub Question
### 2. Multiquery (RAG Fusion)
### 3. Step-back question

#### (Least to most abstract)

# Environment

In [ ]:
# (1) Install required packages (if missing)
! pip install google-generativeai langchain_google_genai chromadb langchain
! pip install langchain_community tiktoken langchainhub

# (2) Import Gemini components
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

In [2]:
from google.colab import userdata # For API Secret

In [5]:
# LangSmith Configuration
import os
from langsmith import traceable   ## To use @traceable on llm calls

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = userdata.get('LANGCHAIN_API_KEY')
os.environ['LANGSMITH_PROJECT']='Rag-from-scratch_P2'
os.environ['GOOGLE_API_KEY']=userdata.get('Gemini_API')

# Gemini configuration
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

In [ ]:
# LangChain Libraries
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Indexing (Same as Phase 1)

In [7]:
# Load Docs

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# *** #

# Split Docs

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
    )

splits = text_splitter.split_documents(docs)

# *** #

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,  # Using Gemini embeddings
)

retriever = vectorstore.as_retriever()

# Prompt

In [9]:
from langchain.prompts import ChatPromptTemplate

# Multi Query : Single query with different perspectives

template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser

# Pass the template prompt to llm to split the question into multiple subquestions(queries).

generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

# Retrieve

In [10]:
from langchain.load import dumps, loads

# Get unique union of retrieved docs
def get_unique_union(documents : list[list]):
  """ Unique union of retrieved docs """

  # Flatten list of lists, and convert each doc to string
  flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]

  # Get unique docs
  unique_docs = list(set(flattened_docs))

  # return
  return [loads(doc) for doc in unique_docs]

In [ ]:
question = "What is task decomposition for llm agents?"

# Retrieve
retrieval_chain = generate_queries | retriever.map() | get_unique_union

docs = retrieval_chain.invoke({"question" : question})